In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/mask-inpainting-checkpoints'

Mounted at /content/drive


In [44]:

!git pull origin yeonholee

print(f"Current working directory: {os.getcwd()}")

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.40 KiB | 2.40 MiB/s, done.
From https://github.com/chaeyoungwon/mask-aware-inpainting
 * branch            yeonholee  -> FETCH_HEAD
   3386051..714f894  yeonholee  -> origin/yeonholee
Updating 3386051..714f894
Fast-forward
 models/pconv_cae.py   | 85 ++++++++++++++++++++++++++++++++++++++++++++++++++-
 models/pconv_layer.py | 64 +++++++++++++++++++++++++++++++++++++-
 2 files changed, 147 insertions(+), 2 deletions(-)
Current working directory: /content/mask-aware-inpainting


In [32]:
!git stash push -m "colab local celeba_dataset edits" datasets/celeba_dataset.py
!git pull origin yeonholee

Saved working directory and index state On yeonholee: colab local celeba_dataset edits
From https://github.com/chaeyoungwon/mask-aware-inpainting
 * branch            yeonholee  -> FETCH_HEAD
Updating f803415..3386051
Fast-forward
 datasets/celeba_dataset.py | 53 +++++++++++++++++++++++++++++++++++++++++++++-
 1 file changed, 52 insertions(+), 1 deletion(-)


## 데이터셋 구성

In [3]:
# kaggle.json 업로드
from google.colab import files
files.upload()  # kaggle.json 선택

# 설치 및 인증
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [4]:
!kaggle datasets download -d jessicali9530/celeba-dataset -p ./mask-aware-inpainting/data/
!cd mask-aware-inpainting/data && unzip celeba-dataset.zip -d celeba_temp

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197604.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197605.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197606.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197607.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197608.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197609.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197610.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197611.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197612.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197613.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197614.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197615.jpg  
  inflating: celeba_temp/img_align_celeba/img_align_celeba/197616.jpg  
  inflating: celeba_temp/img

In [10]:
import os

# 이미지를 올바른 위치로 이동
src = '/content/mask-aware-inpainting/mask-aware-inpainting/data/celeba_temp/img_align_celeba/img_align_celeba'
dst = '/content/mask-aware-inpainting/data/celeba/img_align_celeba'

os.makedirs('/content/mask-aware-inpainting/data/celeba', exist_ok=True)
os.rename(src, dst)
print("완료:", len(os.listdir(dst)), "개")

base = '/content/mask-aware-inpainting/data/celeba'
filenames = sorted(os.listdir(f'{base}/img_align_celeba'))
n = len(filenames)

# list_eval_partition.txt
with open(f'{base}/list_eval_partition.txt', 'w') as f:
    for i, fname in enumerate(filenames):
        split = 0 if i < 162770 else (1 if i < 182637 else 2)
        f.write(f'{fname} {split}\n')

# identity
with open(f'{base}/identity_CelebA.txt', 'w') as f:
    for fname in filenames:
        f.write(f'{fname} 1\n')

# attr (더미)
attrs = ['5_o_Clock_Shadow','Arched_Eyebrows','Attractive','Bags_Under_Eyes','Bald','Bangs','Big_Lips','Big_Nose','Black_Hair','Blond_Hair','Blurry','Brown_Hair','Bushy_Eyebrows','Chubby','Double_Chin','Eyeglasses','Goatee','Gray_Hair','Heavy_Makeup','High_Cheekbones','Male','Mouth_Slightly_Open','Mustache','Narrow_Eyes','No_Beard','Oval_Face','Pale_Skin','Pointy_Nose','Receding_Hairline','Rosy_Cheeks','Sideburns','Smiling','Straight_Hair','Wavy_Hair','Wearing_Earrings','Wearing_Hat','Wearing_Lipstick','Wearing_Necklace','Wearing_Necktie','Young']
with open(f'{base}/list_attr_celeba.txt', 'w') as f:
    f.write(f'{n}\n{" ".join(attrs)}\n')
    for fname in filenames:
        f.write(fname + ' ' + ' '.join(['-1']*40) + '\n')

# bbox / landmarks (더미)
with open(f'{base}/list_bbox_celeba.txt', 'w') as f:
    f.write('image_id x_1 y_1 width height\n')
    for fname in filenames:
        f.write(f'{fname} 0 0 128 128\n')

with open(f'{base}/list_landmarks_align_celeba.txt', 'w') as f:
    f.write(f'{n}\nlefteye_x lefteye_y righteye_x righteye_y nose_x nose_y leftmouth_x leftmouth_y rightmouth_x rightmouth_y\n')
    for fname in filenames:
        f.write(f'{fname} 32 32 96 32 64 64 32 96 96 96\n')

print("메타파일 완료")
print(os.listdir(base))

완료: 202599 개
메타파일 완료
['list_eval_partition.txt', 'identity_CelebA.txt', 'list_bbox_celeba.txt', 'img_align_celeba', 'list_attr_celeba.txt', 'list_landmarks_align_celeba.txt']


In [34]:
# download=False 실제로 저장됐는지 확인
!grep "download" /content/mask-aware-inpainting/datasets/celeba_dataset.py

# 메타데이터 파일 있는지 확인
import os
celeba_dir = '/content/mask-aware-inpainting/data/celeba'
if os.path.exists(celeba_dir):
    print("celeba dir contents:", os.listdir(celeba_dir))
else:
    print("celeba dir does NOT exist!")

    root        : dataset root (CelebA will be downloaded here if needed)
            celeba = CelebA(root=root, split=split, transform=transform, download=False)
celeba dir contents: ['list_eval_partition.txt', 'identity_CelebA.txt', 'list_bbox_celeba.txt', 'img_align_celeba', 'list_attr_celeba.txt', 'list_landmarks_align_celeba.txt']


In [35]:
# 이미지가 어디 있는지 확인
import glob

paths_to_check = [
    '/content/mask-aware-inpainting/data/celeba/img_align_celeba',
    '/content/mask-aware-inpainting/data/celeba/img_align_celeba/img_align_celeba',
    '/content/mask-aware-inpainting/mask-aware-inpainting/data/celeba/img_align_celeba',
]
for p in paths_to_check:
    if os.path.exists(p):
        files = os.listdir(p)
        print(f"✅ {p}: {len(files)} files  (e.g. {files[:3]})")
    else:
        print(f"❌ {p}: not found")

✅ /content/mask-aware-inpainting/data/celeba/img_align_celeba: 202599 files  (e.g. ['139945.jpg', '089570.jpg', '034660.jpg'])
❌ /content/mask-aware-inpainting/data/celeba/img_align_celeba/img_align_celeba: not found
❌ /content/mask-aware-inpainting/mask-aware-inpainting/data/celeba/img_align_celeba: not found


In [37]:
import sys, importlib
sys.path.insert(0, '/content/mask-aware-inpainting')
%cd /content/mask-aware-inpainting

import datasets.celeba_dataset as celeba_dataset
importlib.reload(celeba_dataset)

CelebAInpaintingDataset = celeba_dataset.CelebAInpaintingDataset

/content/mask-aware-inpainting


In [39]:
from datasets.celeba_dataset import CelebAInpaintingDataset

In [40]:
from config import Config
import random, os
import numpy as np
import torch

random.seed(42); np.random.seed(42); torch.manual_seed(42)
cfg = Config()

dataset = CelebAInpaintingDataset(
    root=cfg.data_root,
    split='test',
    image_size=cfg.image_size,
    max_samples=500
)

print(f"Dataset size: {len(dataset)}")

Dataset size: 500


In [41]:
!python3 prepare_fixed_testset.py

Saving 500 samples to ./fixed_testset/  (seed=42) ...
  100/500
  200/500
  300/500
  400/500
  500/500

Saved 500 samples → ./fixed_testset/
Hole ratio — mean: 0.499 | min: 0.018 | max: 0.874
  small  (< 0.3)      : 108 samples
  medium (0.3 – 0.5)  : 125 samples
  large  (0.5 – 0.7)  : 158 samples


In [42]:
# conv_type을 'pconv'으로 설정
import re
with open('config.py', 'r') as f:
    _cfg = f.read()
_cfg = re.sub(r"conv_type\s*=\s*'[^']*'", "conv_type = 'pconv'", _cfg)
with open('config.py', 'w') as f:
    f.write(_cfg)
print("conv_type set to 'pconv'")


conv_type set to 'pconv'


## 학습

In [ ]:
!python3 train.py

device: cuda  |  conv_type: pconv  |  seed: 42
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 528M/528M [00:04<00:00, 132MB/s]
Trainable parameters: 18,038,659
  [1/50] step    0 | loss 2.2096 | valid 0.3109 | hole 0.2378 | perc 9.4378
  [1/50] step   10 | loss 1.3021 | valid 0.0886 | hole 0.1520 | perc 6.0328
  [1/50] step   20 | loss 1.0671 | valid 0.0657 | hole 0.1188 | perc 5.7686
  [1/50] step   30 | loss 0.9170 | valid 0.0669 | hole 0.0993 | perc 5.0899
  [1/50] step   40 | loss 1.0473 | valid 0.0619 | hole 0.1185 | perc 5.4843
  [1/50] step   50 | loss 0.9834 | valid 0.0519 | hole 0.1112 | perc 5.2903
  [1/50] step   60 | loss 1.0513 | valid 0.0399 | hole 0.1221 | perc 5.5757
  [1/50] step   70 | loss 1.0438 | valid 0.0489 | hole 0.1182 | perc 5.7104
  [1/50] step   80 | loss 1.0884 | valid 0.0382 | hole 0.1286 | perc 5.5752
  [1/50] step   90 | loss 0.8772 | valid 0.0597 | hole 0.0982 | perc 4.

In [ ]:
!python evaluate.py checkpoints/pconv/best_model.pth --conv_type pconv --fixed_testset ./fixed_testset


## 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('checkpoints/pconv/training_history_pconv.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('PConv CAE', fontsize=14)

ax1.plot(df['epoch'], df['val_psnr'], color='steelblue')
ax1.set_title('Epoch별 Val PSNR')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('PSNR (dB)'); ax1.grid(True)

ax2.plot(df['epoch'], df['val_ssim'], color='goldenrod')
ax2.set_title('Epoch별 Val SSIM')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('SSIM'); ax2.grid(True)

plt.tight_layout()
plt.show()
print(f'최종 PSNR: {df["val_psnr"].iloc[-1]:.2f} dB  |  최종 SSIM: {df["val_ssim"].iloc[-1]:.4f}')
